# ⏳ 05 — Shock Persistence by Shock Type

How long does a shock last, and does it depend on its type? Hourly Binance data, 30 coins, 2020–2026.

1. Event windows around the catalogued crises.
2. Persistence in hours: how long until a shocked coin's volatility is back to normal.
3. Persistence by shock type: market-wide vs partial vs idiosyncratic, crash vs pump; re-shocks, followers,
   continuation or reversal.
4. Implications for the models: how well the contagion models do for each type of source shock.

**Definitions** (`cryptograph.analysis.persistence`). A shock is |z| > 2.29 with z = hourly return / trailing 30-day
volatility (the project's leakage-free shock). Only **fresh** shocks are studied (no shock of the same coin in the
previous 24 h), so one turbulent episode counts once. Type by **breadth**, the share of the other coins shocking in
the same hour: **market-wide** ≥ 30%, **partial** 10–30%, **idiosyncratic** < 10%. Intervals resample whole days
(500 times), because shocks on the same day are not independent.

In [1]:
import sys, json
from pathlib import Path
src_path = str(Path("../src").resolve())
if src_path not in sys.path:
    sys.path.insert(0, src_path)

import torch  # keep torch before pandas (Windows DLL order)
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

pio.renderers.default = "plotly_mimetype+notebook_connected"
pio.templates.default = "plotly_dark"
V2 = "../data/processed/v2"

from cryptograph.analysis.event_study import event_window

hp = torch.load(f"{V2}/1h/panel.pt", weights_only=False)
times = pd.to_datetime(hp["times"])
model = hp["model_coins"]
close = pd.read_parquet("../data/interim/hourly/master_1h.parquet")["close"].unstack("coin").reindex(index=times, columns=model)
r = np.log(close).diff()
events = pd.read_csv("../data/external/events.csv", parse_dates=["date"])
events = events[events["date"] >= "2020-01-08"]

## 1. Event windows (hourly)

In [2]:
rows = []
for _, e in events.iterrows():
    w = event_window(r, e["date"], pre=24, post=72, freq="h")
    rows.append(pd.DataFrame({"hour": w.index, "median cumulative return": w.median(axis=1).values,
                              "event": f"{e['date'].date()} {e['name'][:28]}"}))
ew = pd.concat(rows)
fig = px.line(ew, x="hour", y="median cumulative return", color="event",
              title="Median cumulative log return across the 30 coins, 24 h before to 72 h after each event (hour 0 = 00:00 UTC of the event date)")
fig.add_vline(x=0, line_dash="dash", line_color="#ff5252")
fig.update_layout(height=460, yaxis_tickformat=".0%")
fig.show()
display(ew[ew["hour"].isin([0, 6, 24, 72])].pivot(index="event", columns="hour", values="median cumulative return")
        .map("{:+.1%}".format))

hour,0,6,24,72
event,,,,
2020-03-12 COVID Black Thursday,-0.7%,-9.3%,-53.3%,-42.3%
2021-05-19 China bans financial institu,-0.5%,-13.6%,-51.5%,-46.2%
2022-05-09 Terra UST depeg / LUNA colla,+1.3%,-2.4%,-15.6%,-32.2%
2022-06-13 Celsius withdrawal freeze /,-3.3%,-7.5%,-12.2%,+0.8%
2022-11-08 FTX collapse,-0.4%,-6.2%,-15.2%,-16.6%
2023-03-10 SVB failure / USDC depeg,-1.1%,-1.6%,+1.3%,+9.5%
2024-01-10 US spot BTC ETF approval,+0.5%,-0.2%,+9.0%,+4.1%
2024-08-05 Yen carry-trade unwind,-4.6%,-16.2%,-3.3%,-5.3%
2025-10-10 Oct-10 liquidation cascade,+0.1%,+0.6%,-20.6%,-15.2%


*What this shows:* the median coin's path around each crisis, hour by hour. Event dates are calendar days
(hour 0 is midnight UTC), so some moves start before hour 0; the exact hour of each shock is located in the
training phase from the shock labels. This is the starting point for measuring persistence in hours.

## 2. Persistence in hours

In [3]:
from cryptograph.analysis.persistence import abs_z_profile, block_bootstrap_ci, persistence_table, TYPE_ORDER

gi = torch.load(f"{V2}/1h/tgn_events.pt", weights_only=False)["graph_index"]
coins = list(hp["graph_coins"])
r1 = hp["X_raw"][:, gi, hp["feature_names"].index("log_return")].numpy().astype(float)
z1 = hp["shock_z"].numpy()[:, gi].astype(float)
with np.errstate(divide="ignore", invalid="ignore"):
    sig = np.where(np.abs(z1) > 1e-9, r1 / z1, np.nan)
sh, va, act = (hp[k].numpy()[:, gi] for k in ("shock_today", "shock_today_mask", "active"))
ps = persistence_table(r1, z1, sig, sh, va, act, times, coins)
ps["period"] = np.array(["train", "val", "test", "other"])[np.clip(hp["split_id"].numpy()[ps["t"]], 0, 3)]
for c in ("reshock_24h",):
    ps[c] = ps[c].astype(float)
print(f"{len(ps):,} fresh shocks ({sh.sum():,} shock hours in total)")
display(ps.groupby(["type", "direction"]).size().unstack().loc[TYPE_ORDER])
prof = abs_z_profile(z1, ps, coins, by="type")
fig = px.line(prof, x="hour", y="mean_abs_z", color="type", category_orders={"type": TYPE_ORDER},
              title="Mean |z| of the shocked coin around a fresh shock (hour 0 = shock)")
fig.add_hline(y=float(np.nanmean(np.abs(z1))), line_dash="dot", annotation_text="all-hours average")
fig.update_layout(height=430)
fig.show()
h = block_bootstrap_ci(ps, "hours_to_normal", ["type", "direction"])
display(h.set_index(["type", "direction"]).loc[TYPE_ORDER].round(1))
print("share of shocks with volatility back to normal within 6 / 24 / 72 h:",
      [round(float((ps["hours_to_normal"] <= x).mean()), 3) for x in (6, 24, 72)])

18,171 fresh shocks (57,792 shock hours in total)


direction,crash,pump
type,,
market-wide,6275,3068
partial,1320,1808
idiosyncratic,1653,4047


n_shocks  n_days  median  ci_low  ci_high
type          direction                                           
market-wide   crash          6268     700    10.0     9.0     11.0
              pump           3066     373     9.0     8.0      9.0
partial       crash          1319     446     9.0     8.0     10.0
              pump           1807     546     9.0     9.0     10.0
idiosyncratic crash          1653    1011     8.0     8.0      9.0
              pump           4041    1668    10.0    10.0     11.0

share of shocks with volatility back to normal within 6 / 24 / 72 h: [0.284, 0.866, 0.986]


*What this shows:* **Shocks fade within hours, not days.** Across 18,171 fresh shocks, a coin's volatility is back to its
pre-shock level after a median of **8–10 hours** for every type. 28% have normalised within 6 hours, 87% within
24 hours and 99% within 72 hours. The differences between types are small but real: idiosyncratic crashes fade
fastest (median 8 h), while idiosyncratic pumps and market-wide crashes last longest (10 h). The |z| profile shows the
same: every type starts from the same calm level (mean |z| ≈ 0.62 in the day before, below the all-hours average of
0.72), spikes at hour 0 (3.5 for market-wide shocks, about 3.0 for the others), is down to about 1.1 one hour later
and about 0.8 after 6 hours. It then stays slightly above its pre-shock level for the rest of the day, which is the
re-shock clustering of section 3. The persistence
is the same in all three periods (9–10 h in train, validation and test), so it is a stable property of this
market and not of one regime. Over a day, the moves of the catalogued crises (section 1) come from repeated shocks,
not from one long shock.

## 3. Persistence by shock type

Re-shock: another shock of the same coin within 24 h. Followers: other coins shocking within 6 h. Continuation:
signed cumulative return after the shock in units of the coin's volatility (> 0 = the move continues, < 0 = it
reverses). Means with day-block 95% intervals.

In [4]:
tabs = []
for col in ("reshock_24h", "followers_6h", "continuation_6h", "continuation_24h"):
    t = block_bootstrap_ci(ps, col, ["type", "direction"], stat="mean")
    t["measure"] = col
    tabs.append(t)
tb = pd.concat(tabs)
tb["value"] = tb.apply(lambda x: f"{x['mean']:+.2f} [{x['ci_low']:+.2f}, {x['ci_high']:+.2f}]", axis=1)
display(tb.pivot_table(index=["type", "direction"], columns="measure", values="value", aggfunc="first").loc[TYPE_ORDER])
st = ps.groupby(["period", "type"])["hours_to_normal"].median().unstack().reindex(["train", "val", "test"])[TYPE_ORDER]
print("median hours to normal, by period (stability check):")
display(st)

measure                      continuation_24h       continuation_6h  \
type          direction                                               
market-wide   crash      +0.30 [-0.20, +0.83]  +0.17 [-0.15, +0.50]   
              pump       +0.60 [-0.06, +1.32]  +0.42 [+0.07, +0.80]   
partial       crash      -0.06 [-0.68, +0.53]  +0.08 [-0.34, +0.54]   
              pump       -0.03 [-0.60, +0.54]  +0.19 [-0.08, +0.45]   
idiosyncratic crash      -0.20 [-0.59, +0.18]  -0.13 [-0.29, +0.02]   
              pump       +0.34 [+0.07, +0.64]  +0.12 [+0.01, +0.26]   

measure                          followers_6h           reshock_24h  
type          direction                                              
market-wide   crash      +8.11 [+7.32, +8.87]  +0.61 [+0.58, +0.64]  
              pump       +7.22 [+6.24, +8.25]  +0.58 [+0.54, +0.62]  
partial       crash      +6.19 [+5.43, +7.02]  +0.61 [+0.56, +0.65]  
              pump       +6.19 [+5.54, +6.89]  +0.65 [+0.61, +0.68]  
idiosyncratic crash      +5.05 [+4.69, +5.51]  +0.55 [+0.52, +0.58]  
              pump       +4.53 [+4.27, +4.80]  +0.63 [+0.61, +0.65]

median hours to normal, by period (stability check):


type,market-wide,partial,idiosyncratic
period,,,
train,10.0,10.0,10.0
val,9.0,9.0,10.0
test,9.0,9.0,9.0


*What this shows:* **Three things differ by type, one does not.**
- **Spillover differs most:** a market-wide shock is followed by shocks of **8.1** other coins within 6 hours (crash)
  vs **4.5–5.1** after an idiosyncratic shock. All intervals separate.
- **Re-shocks are the norm:** 55–65% of fresh shocks are followed by another shock of the same coin within 24 hours.
  Shocks cluster, which is why recent volatility is such a strong input for every model. Idiosyncratic crashes
  re-shock least (55%).
- **Direction is mostly not predictable.** Most continuation intervals include zero. The exceptions are pumps:
  idiosyncratic pumps keep rising (+0.34σ over 24 h, interval above zero), as do market-wide pumps over 6 hours
  (+0.42σ). Crashes show no reliable continuation or reversal, which matches notebook 01 (returns are not
  predictable).
- **What does not differ:** how long volatility stays elevated (section 2).

## 4. Implications for the models: contagion predictability by source-shock type

The final TGN and the gradient-boosting baseline scored every test event; here the test events are split by the type
of the *source* shock (breadth and direction at the hour of the event). AUPRC is read against the positive rate of
that group (= random guessing).

In [5]:
from cryptograph.training.metrics import auprc

def ens(exp):
    import glob
    ps_ = [torch.load(f, weights_only=False) for f in sorted(glob.glob(f"../runs/*_{exp}_*/predictions_test.pt"))]
    return torch.stack([p["prob"].float() for p in ps_]).mean(0).numpy(), ps_[0]

tg, ref = ens("tgn_1h_final_v2")
gb, ref2 = ens("gbm_1h_final")
assert np.array_equal(ref["position"].numpy(), ref2["position"].numpy())
tt, src = ref["t"].numpy(), ref["src"].numpy()
from cryptograph.analysis.persistence import shock_breadth, shock_type
br = shock_breadth(sh & va, act & va)
typ = shock_type(br[tt, src])
direction = np.where(r1[tt, src] < 0, "crash", "pump")
y, m = ref["y"].numpy(), ref["mask"].numpy()
K = list(ref["k_hours"])
rows = []
for ty in TYPE_ORDER:
    for dr in ("crash", "pump"):
        g = (typ == ty) & (direction == dr)
        for i, k in enumerate(K):
            mk = g & m[:, i]
            if mk.sum() < 100:
                continue
            rows.append({"type": ty, "direction": dr, "k_hours": k, "events": int(mk.sum()),
                         "positive rate": y[mk, i].mean(), "TGN": auprc(y[mk, i], tg[mk, i]),
                         "boosting": auprc(y[mk, i], gb[mk, i])})
bt = pd.DataFrame(rows)
bt["TGN lift"] = bt["TGN"] / bt["positive rate"]
display(bt[bt["k_hours"].isin([1, 6, 24])].set_index(["type", "direction", "k_hours"]).round(3))
fig = px.bar(bt[bt["k_hours"] == 6], x="type", y="TGN lift", color="direction", barmode="group",
             category_orders={"type": TYPE_ORDER}, title="TGN, k = 6 h: AUPRC / positive rate by source-shock type (test)")
fig.update_layout(height=400)
fig.show()

events  positive rate    TGN  boosting  \
type          direction k_hours                                           
market-wide   crash     1        257660          0.163  0.245     0.205   
                        6        257660          0.397  0.536     0.509   
                        24       257660          0.700  0.786     0.785   
              pump      1        165497          0.118  0.225     0.201   
                        6        165497          0.338  0.485     0.461   
                        24       165483          0.624  0.748     0.726   
partial       crash     1         54662          0.088  0.130     0.103   
                        6         54662          0.301  0.426     0.400   
                        24        54662          0.632  0.729     0.702   
              pump      1         85970          0.070  0.129     0.111   
                        6         85970          0.271  0.378     0.372   
                        24        85970          0.625  0.731     0.725   
idiosyncratic crash     1         83380          0.058  0.114     0.117   
                        6         83380          0.218  0.312     0.329   
                        24        83377          0.554  0.629     0.651   
              pump      1        148618          0.043  0.086     0.088   
                        6        148618          0.208  0.310     0.319   
                        24       148613          0.543  0.645     0.648   

                                 TGN lift  
type          direction k_hours            
market-wide   crash     1           1.500  
                        6           1.351  
                        24          1.124  
              pump      1           1.912  
                        6           1.435  
                        24          1.199  
partial       crash     1           1.479  
                        6           1.416  
                        24          1.154  
              pump      1           1.849  
                        6           1.396  
                        24          1.170  
idiosyncratic crash     1           1.959  
                        6           1.433  
                        24          1.136  
              pump      1           1.995  
                        6           1.487  
                        24          1.188

*What this shows:* **The graph model earns its place exactly where contagion is network-wide.** After market-wide and partial
shocks the TGN beats gradient boosting at every horizon: after a market-wide crash, 0.245 vs 0.205 at 1 hour and 0.536
vs 0.509 at 6 hours; after a partial crash, 0.130 vs 0.103 at 1 hour. After **idiosyncratic** shocks gradient
boosting is level or slightly ahead (0.117 vs 0.114 at 1 hour, 0.329 vs 0.312 at 6 hours): when one coin moves on
its own news, the network adds nothing and the coins' own state is all there is. Contagion is also far more likely
after market-wide crashes (positive rate 16% at 1 hour vs 4–6% after idiosyncratic shocks), and relative to that
base rate the TGN does well across all types (about 1.5–2× random at 1 hour).
**Implications:** (1) the main horizons for alerts are the first 6–24 hours, since volatility is normal again after
about half a day; (2) a deployed system could use the TGN after broad shocks and a simpler feature model after
isolated ones; (3) persistence does not need its own model target: it is short and stable, so the contagion labels
at 1–24 hours already cover it.

## Key Takeaways

- **Shocks are short-lived:** a coin's volatility is back to normal after a median of 8–10 hours (87% within a day),
  for every shock type and in every period.
- **Shocks cluster:** 55–65% are followed by another shock of the same coin within 24 hours.
- **Spillover depends on type:** a market-wide shock is followed by about 8 other coins shocking within 6 hours,
  an idiosyncratic one by about 5.
- **Direction is mostly unpredictable,** except that pumps tend to continue for a while.
- **The TGN beats gradient boosting after market-wide and partial shocks, but not after idiosyncratic ones.** The
  temporal graph helps when the shock is network-wide, which is what contagion means.
- **For the models:** contagion horizons up to 24 hours cover the whole life of a typical shock; no separate
  persistence model is needed.